## Bronze Setup for Dev
Since Free Edition can't ingest files from Kafka streaming, I am creating a small notebook that will seed the pipeline with sample files for development purposes.

In [0]:
# Configuration
# Creating widgets


dbutils.widgets.text("catalog", "workspace", "Unity Catalog")
dbutils.widgets.text("bronze_schema", "live_transit_monitor", "Bronze schema")

CATALOG = dbutils.widgets.get("catalog")          # workspace na Free Edition
SCHEMA  = dbutils.widgets.get("bronze_schema")
SEED = f"/Volumes/{CATALOG}/{SCHEMA}/project_volume/dev_seed"


In [0]:
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA}")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {CATALOG}.{SCHEMA}.project_volume")

In [0]:
REPO = "file:/Workspace/Repos/gabrielajaniszewska@translite.pl/live-transit-monitor/sample_data"
files = [
    "gps_20260727_210042.json",   
    "gps_20260729_173022.json",
    "gps_20260729_180841.json",
    "gps_bad_rows.json",          # bad records on purpose for testing
]
for f in files:
    dbutils.fs.cp(f"{REPO}/{f}", f"{SEED}/{f}")

display(dbutils.fs.ls(SEED))

In [0]:
# The same StructType as in 02_gps_consumer
from pyspark.sql.types import StructType, StringType, IntegerType, FloatType
event_schema = (StructType()
    .add("generated", StringType()).add("routeShortName", StringType())
    .add("tripId", IntegerType()).add("routeId", IntegerType())
    .add("headsign", StringType()).add("vehicleCode", StringType())
    .add("vehicleService", StringType()).add("vehicleId", IntegerType())
    .add("speed", IntegerType()).add("direction", IntegerType())
    .add("delay", IntegerType()).add("scheduledTripStartTime", StringType())
    .add("lat", FloatType()).add("lon", FloatType())
    .add("gpsQuality", IntegerType()).add("lastUpdate", StringType()))

In [0]:
from pyspark.sql.functions import to_timestamp, current_timestamp, lit

bronze = (spark.read.schema(event_schema).json(SEED)
    .withColumn("event_time",   to_timestamp("generated"))
    .withColumn("_source",      lit("dev-sample"))
    .withColumn("ingestion_ts", current_timestamp()))

(bronze.write.format("delta").mode("overwrite").option("overwriteSchema","true")
    .saveAsTable(f"{CATALOG}.{SCHEMA}.gps_data"))

### Checking the bronze data

In [0]:
b = spark.read.table(f"{CATALOG}.{SCHEMA}.gps_data")
print("rows:", b.count())
print("event_time nulls:", b.filter("event_time IS NULL").count())

# Checking if bad records are there:
b.filter("vehicleId IN (999001, 999002, 999003)").select(
    "vehicleId","headsign","lat","lon").show()

# Checking if there are duplicated vehicleId for dedup:
from pyspark.sql import functions as F
b.groupBy("vehicleId").count().filter("count > 1").show(5)